# 11.10b Projekt: Eigenes Sprachmodell trainieren

In 11.0 (KI-Kickstart) hast du gelernt, wie ein Sprachmodell arbeitet: Es berechnet Logits, macht daraus mit **Softmax** (und einer **Temperatur**) Wahrscheinlichkeiten für das nächste Token, wird mit dem **Verlust** $L = -\ln p_{\text{richtig}}$ trainiert und erzeugt Text **autoregressiv**, also Token für Token. In diesem Notebook baust du so ein Modell **selbst** und trainierst es:

> Ein **Zeichen-Sprachmodell** lernt aus rund 400 echten sächsischen Ortsnamen, welcher Buchstabe als Nächstes kommt, und erfindet danach neue Ortsnamen, die sächsisch klingen (oder auch nicht).

Im Notebook zu 11.10 hast du schon Wortpaare in einem `dict` gezählt. Hier gehst du drei Schritte weiter: Du rechnest mit **Matrizen** (numpy), du **misst**, wie gut ein Modell ist (Verlust auf Trainings- und Testdaten), und du ersetzt das Zählen durch ein kleines **neuronales Netz**, das mit **Gradientenabstieg** trainiert wird, so wie die großen Modelle.

| Begriff aus 11.0 | in diesem Notebook |
|---|---|
| Token, Token-ID | ein einzelnes Zeichen (Buchstabe) oder das Sonderzeichen `.`; jedes hat eine Nummer 0 … V−1 |
| Vokabular | alle Zeichen, die in den Ortsnamen vorkommen, plus `.` (V ≈ 30) |
| Logits | eine Zeile der Gewichtsmatrix `W` (Abschnitt 3) |
| Softmax mit Temperatur T | genau wie in 11.0, nur mit numpy |
| Verlust $-\ln p$ | genau wie in 11.0, gemittelt über alle Zeichenpaare |
| Training | Zählen (Abschnitt 2) bzw. Gradientenabstieg (Abschnitt 3) |
| Kontext (Attention, Kontextfenster) | nur das **letzte** Zeichen (Bigramm-Modell); in Zusatz 5d drei Zeichen |

## So arbeitest du

- **Erklärungen** stehen vollständig in den Textzellen. Lies sie, bevor du Code schreibst.
- **Aufgaben mit Code** erkennst du an `# TODO`. Ersetze `...` bzw. die Zeile `raise NotImplementedError(...)` durch deinen Code.
- **Tipps** sind eingeklappt (▸ Tipp). Öffne sie erst, wenn du nicht weiterkommst.
- **Testzellen** (`# TEST`) prüfen deinen Code mit `assert`. Führe sie aus, ändere sie nicht. Erst wenn „bestanden“ erscheint, geht es weiter. Eine Fehlermeldung `AssertionError` sagt dir, was noch nicht stimmt.
- **Denkaufgaben** beantwortest du in einer eigenen Markdown-Zelle direkt darunter (*Einfügen → Zelle darunter*, Zelltyp „Markdown“).
- Abschnitt 5 enthält **Erweiterungen** zur Auswahl, Abschnitt 6 die **Reflexion**.

**Technik:** Öffne das Notebook in JupyterLite unter [jupyterlite.inf7.eu](https://jupyterlite.inf7.eu) (Datei in den Dateibereich ziehen) und führe die Zellen der Reihe nach mit `Umschalt + Enter` aus. JupyterLite speichert nur im Browser: Lade das Notebook am Ende jeder Stunde herunter (Rechtsklick → *Download*) und lege es im **Gruppenrepository auf git.inf7.eu** ab (Commit mit aussagekräftiger Nachricht, z. B. „Abschnitt 2: Zählmodell fertig, Tests grün“). Wenn ihr in der Gruppe arbeitet, legt fest, wer welchen Abschnitt bearbeitet, und führt die Stände im Repository zusammen.

**Schreibweise:** Wie im Kurs üblich stehen in Python-Namen keine Umlaute (`naechstes`, `zaehle_echte`). Python rechnet in Kommazahlen mit Punkt: `0.5` bedeutet 0,5.

In [ ]:
import math
import random
import numpy as np

rng = np.random.default_rng(2026)   # Zufallsgenerator zum Ziehen von Zeichen (fester Startwert)
print("numpy-Version:", np.__version__)

## 1. Daten: sächsische Ortsnamen

Die Trainingsdaten stehen direkt in der nächsten Zelle: rund 400 echte Namen von Städten, Gemeinden und einigen Ortsteilen aus allen Teilen Sachsens. Damit das Vokabular klein bleibt, sind nur Namen aus **einem Wort** ohne Bindestrich aufgenommen (also kein „Bad Schandau“ und kein „Annaberg-Buchholz“).

Ein Sprachmodell lernt nur aus seinen Daten. Schau dir die Liste deshalb kurz an: Welche Endungen fallen dir auf? Viele sächsische Ortsnamen haben sorbisch-slawischen Ursprung (z. B. auf *-itz*, *-witz*, *-au*), andere deutschen (z. B. auf *-dorf*, *-berg*, *-bach*, *-hain*, *-walde*).

In [ ]:
orte_roh = """
Leipzig Markkleeberg Markranstädt Taucha Schkeuditz Borna Grimma Wurzen Naunhof Brandis
Böhlen Zwenkau Pegau Groitzsch Frohburg Geithain Colditz Trebsen Machern Bennewitz
Thallwitz Borsdorf Großpösna Belgershain Parthenstein Kitzscher Neukieritzsch Rötha
Gohlis Lindenau Plagwitz Connewitz Probstheida Paunsdorf Engelsdorf Liebertwolkwitz Lützschena Wiederitzsch
Möckern Stötteritz Reudnitz Schleußig Eutritzsch
Delitzsch Eilenburg Torgau Oschatz Mügeln Dahlen Belgern Schildau Dommitzsch Trossin
Arzberg Beilrode Cavertitz Liebschützberg Wermsdorf Laußig Doberschütz Jesewitz Krostitz
Schönwölkau Wiedemar Rackwitz Löbnitz Zschepplin Mockrehna Elsnig
Dresden Blasewitz Loschwitz Pillnitz Striesen Laubegast Niedersedlitz Lockwitz Prohlis Löbtau
Cotta Gorbitz Briesnitz Cossebaude Pieschen Kaditz Klotzsche Hellerau Langebrück Weißig Bühlau
Meißen Radebeul Coswig Moritzburg Weinböhla Niederau Klipphausen Käbschütztal Lommatzsch
Nossen Riesa Strehla Zeithain Glaubitz Nünchritz Hirschstein Stauchitz Großenhain Priestewitz
Ebersbach Lampertswalde Schönfeld Thiendorf Radeburg Gröditz Wülknitz Röderaue Nauwalde
Pirna Heidenau Dohna Müglitztal Bahretal Dohma Struppen Königstein Rathen Hohnstein
Stolpen Sebnitz Neustadt Lohmen Rathmannsdorf Gohrisch Liebstadt Glashütte Altenberg Geising
Dippoldiswalde Hermsdorf Klingenberg Kreischa Bannewitz Freital Rabenau Tharandt Wilsdruff
Dorfhain Mohorn Wehlen
Chemnitz Rabenstein Siegmar Grüna Einsiedel Ebersdorf
Freiberg Mittweida Döbeln Hainichen Frankenberg Flöha Augustusburg Leubsdorf Eppendorf Oederan
Rochlitz Burgstädt Penig Lunzenau Geringswalde Leisnig Waldheim Hartha Roßwein
Großschirma Frauenstein Sayda Kriebstein Seelitz Erlau Altmittweida Taura
Mühlau Lichtenau Niederwiesa Claußnitz Hartmannsdorf Wechselburg Großweitzschen Striegistal Reinsberg Halsbrücke
Oberschöna Mulda Dorfchemnitz Neuhausen Lichtenberg Weißenborn Ostrau
Aue Schneeberg Bockau Zschorlau Eibenstock Schönheide Stützengrün Lauter Breitenbrunn
Crottendorf Sehmatal Bärenstein Königswalde Mildenau Tannenberg Geyer Elterlein Zwönitz
Grünhain Jahnsdorf Gornsdorf Auerbach Burkhardtsdorf Thalheim Oelsnitz Hohndorf Niederdorf
Lugau Neukirchen Stollberg Niederwürschnitz Gelenau Großolbersdorf Großrückerswalde Schlettau Scheibenberg
Oberwiesenthal Jöhstadt Schwarzenberg Johanngeorgenstadt Zschopau Drebach Amtsberg Gornau Thum Ehrenfriedersdorf
Wolkenstein Marienberg Pockau Lengefeld Olbernhau Seiffen Deutschneudorf Heidersdorf Zöblitz
Plauen Adorf Markneukirchen Klingenthal Schöneck Falkenstein Rodewisch Treuen Lengenfeld Reichenbach
Netzschkau Elsterberg Pausa Mühltroff Weischlitz Bösenbrunn Eichigt Triebel Tirpersdorf
Werda Bergen Theuma Neuensalz Neumark Heinsdorfergrund Pöhl Steinberg Ellefeld Grünbach
Muldenhammer Mühlental
Zwickau Werdau Crimmitschau Meerane Glauchau Waldenburg Lichtenstein Oberlungwitz Gersdorf Callenberg
Remse Langenweißbach Wildenfels Hartenstein Kirchberg Hirschfeld Crinitzberg Lichtentanne Mülsen Reinsdorf
Fraureuth Langenbernsdorf Dennheritz Niederfrohna
Bautzen Kamenz Hoyerswerda Bischofswerda Radeberg Pulsnitz Königsbrück Großröhrsdorf Wittichenau Elstra
Lauta Arnsdorf Wachau Ohorn Steina Großnaundorf Haselbachtal Schwepnitz Neukirch Steinigtwolmsdorf
Wilthen Gaußig Kubschütz Malschwitz Radibor Puschwitz Nebelschütz Crostwitz Räckelwitz Göda
Großpostwitz Obergurig Cunewalde Sohland Schirgiswalde Großdubrau Königswartha Lohsa Elsterheide
Spreetal Burkau Rammenau Großharthau Hochkirch
Görlitz Zittau Löbau Niesky Weißwasser Herrnhut Bernstadt Ostritz Rothenburg Weißenberg
Kottmar Krauschwitz Boxberg Trebendorf Schleife Rietschen Hähnichen Horka
Kodersdorf Mücka Hohendubrau Waldhufen Vierkirchen Königshain Großschönau Hainewalde Mittelherwigsdorf Olbersdorf
Oybin Jonsdorf Seifhennersdorf Leutersdorf Oderwitz Dürrhennersdorf Lawalde Beiersdorf Großschweidnitz
Schöpstal Strahwalde
"""

orte = sorted(set(orte_roh.lower().split()))   # Kleinbuchstaben, jeder Name nur einmal
print(len(orte), "Ortsnamen, z. B.", orte[:8])
print("durchschnittliche Länge:", round(sum(len(o) for o in orte) / len(orte), 1), "Zeichen")

# häufigste Endungen (letzte 4 Buchstaben)
endungen = {}
for o in orte:
    endungen[o[-4:]] = endungen.get(o[-4:], 0) + 1
haeufigste = sorted(endungen, key=endungen.get, reverse=True)[:8]
print("häufigste Endungen:", [(e, endungen[e]) for e in haeufigste])

### 1.1 Vokabular: Zeichen werden Zahlen

In 11.0 waren Tokens Wortteile. Unser Modell arbeitet mit **einzelnen Zeichen** als Tokens. Dazu kommt ein Sonderzeichen `.`, das gleichzeitig **Start** und **Ende** eines Namens markiert. Aus „pirna“ wird so die Zeichenfolge `.pirna.`:

- Was auf das erste `.` folgt, sagt dem Modell, **wie Namen anfangen**.
- Wann ein `.` folgt, sagt ihm, **wann ein Name zu Ende ist**.

Das Vokabular besteht aus `.` (ID 0) und allen vorkommenden Buchstaben in sortierter Reihenfolge (IDs 1 bis V−1).

In [ ]:
zeichen = sorted(set("".join(orte)))   # alle vorkommenden Buchstaben
vokabular = ["."] + zeichen            # "." bekommt die ID 0
V = len(vokabular)
print("Vokabular:", "".join(vokabular))
print("V =", V)

**Aufgabe 1 (Code):** Ein neuronales Netz rechnet nur mit Zahlen. Erstelle zwei Abbildungen (als `dict`):

- `zeichen_zu_id`: Zeichen → ID, also `"."` → 0, `"a"` → 1, …
- `id_zu_zeichen`: ID → Zeichen, also die Umkehrung.

<details><summary>Tipp</summary>

Die ID eines Zeichens ist seine Position in der Liste `vokabular`. Mit `for i, z in enumerate(vokabular):` bekommst du Position und Zeichen gleichzeitig. Lege vorher zwei leere `dict`s an und trage in der Schleife jeweils einen Eintrag in beide ein.

</details>

In [ ]:
# TODO (Aufgabe 1): Abbildungen Zeichen -> ID und ID -> Zeichen erstellen
zeichen_zu_id = ...
id_zu_zeichen = ...

print(zeichen_zu_id)

In [ ]:
# TEST zu Aufgabe 1 (nur ausführen)
assert isinstance(zeichen_zu_id, dict), "zeichen_zu_id ist noch kein dict (TODO in Aufgabe 1)"
assert len(zeichen_zu_id) == V, "zeichen_zu_id muss genau V Einträge haben"
assert zeichen_zu_id["."] == 0, "'.' muss die ID 0 haben"
for z in vokabular:
    assert id_zu_zeichen[zeichen_zu_id[z]] == z, f"Hin- und Rückweg passen nicht für {z!r}"
print("Alle Tests zu Aufgabe 1 bestanden.")

### 1.2 Trainings- und Testdaten

Wie in 11.3 (KI-Modelle optimieren) teilen wir die Daten auf: **80 %** der Namen zum Trainieren, **20 %** zum Testen. Die Testnamen bekommt das Modell beim Training nie zu sehen. Nur so können wir prüfen, ob es allgemeine **Muster** gelernt hat oder die Trainingsdaten nur **auswendig** kann (Overfitting).

Der feste Startwert (Seed) 42 sorgt dafür, dass alle im Kurs dieselbe Aufteilung bekommen und ihre Ergebnisse vergleichen können.

In [ ]:
mischer = random.Random(42)       # eigener Zufallsgenerator mit festem Seed
gemischt = list(orte)
mischer.shuffle(gemischt)
grenze = int(0.8 * len(gemischt))
train_namen = gemischt[:grenze]
test_namen = gemischt[grenze:]
print(len(train_namen), "Trainingsnamen, z. B.", train_namen[:5])
print(len(test_namen), "Testnamen, z. B.", test_namen[:5])

### 1.3 Trainingspaare (Bigramme)

Ein Sprachmodell lernt **selbstüberwacht** (11.0, Abschnitt 6): Der Text liefert die richtigen Antworten selbst. Aus `.pirna.` entstehen 6 Paare (Kontext → nächstes Zeichen):

| Kontext `x` | `.` | `p` | `i` | `r` | `n` | `a` |
|---|---|---|---|---|---|---|
| nächstes Zeichen `y` | `p` | `i` | `r` | `n` | `a` | `.` |

Ein Name mit n Buchstaben liefert also n + 1 Paare. Wir speichern alle Paare als zwei numpy-Arrays mit IDs: `xs` (Kontexte) und `ys` (jeweils richtiges nächstes Zeichen).

**Aufgabe 2 (Code):** Ergänze die Funktion `bigramme(namen)`.

<details><summary>Tipp</summary>

Gehe mit `for i in range(len(folge) - 1):` alle Positionen durch. Das Paar an Position i ist `folge[i]` und `folge[i + 1]`. Hänge mit `xs.append(...)` bzw. `ys.append(...)` jeweils die **ID** an (über `zeichen_zu_id`), nicht das Zeichen selbst.

</details>

In [ ]:
def bigramme(namen):
    """Liefert zwei numpy-Arrays xs, ys: xs[i] ist die ID eines Zeichens,
    ys[i] die ID des Zeichens, das im Namen direkt darauf folgt."""
    xs = []
    ys = []
    for name in namen:
        folge = "." + name + "."
        # TODO (Aufgabe 2): alle benachbarten Zeichenpaare von folge durchgehen
        #                   und deren IDs an xs (erstes Zeichen) und ys (zweites Zeichen) anhängen
        ...
    return np.array(xs, dtype=int), np.array(ys, dtype=int)

xs_train, ys_train = bigramme(train_namen)
xs_test, ys_test = bigramme(test_namen)
print(len(xs_train), "Trainingspaare,", len(xs_test), "Testpaare")

In [ ]:
# TEST zu Aufgabe 2 (nur ausführen)
xs_p, ys_p = bigramme(["pirna"])
assert len(xs_p) == 6 and len(ys_p) == 6, "für 'pirna' muss es genau 6 Paare geben"
assert [id_zu_zeichen[int(i)] for i in xs_p] == list(".pirna"), "Kontexte für 'pirna' falsch"
assert [id_zu_zeichen[int(i)] for i in ys_p] == list("pirna."), "nächste Zeichen für 'pirna' falsch"
assert len(xs_train) == sum(len(n) + 1 for n in train_namen), "jeder Name liefert len(name) + 1 Paare"
print("Alle Tests zu Aufgabe 2 bestanden.")

## 2. Das Zählmodell

### 2.1 Zählmatrix und Wahrscheinlichkeiten

Das erste Modell funktioniert wie das Bigramm-Modell aus 11.10 (Beispiel 2), nur mit Zeichen statt Wörtern und in einer **Matrix** statt in einem `dict`:

- `N` ist eine V×V-Matrix. `N[i, j]` gibt an, **wie oft** Zeichen j in den Trainingsnamen direkt auf Zeichen i folgt.
- Zeile i von `N` beschreibt also alles, was das Modell über „was kommt nach Zeichen i?“ weiß.

Teilt man jede Zeile durch ihre Summe, erhält man Wahrscheinlichkeiten. Dabei gibt es ein Problem: Ein Paar, das im Training **nie** vorkam, bekäme die Wahrscheinlichkeit 0, und kommt es in einem Testnamen vor, wäre der Verlust $-\ln 0 = \infty$. Deshalb **glätten** wir: Wir tun so, als hätten wir jedes Paar einmal zusätzlich gesehen (+1-Glättung):

$$P_{ij} = \frac{N_{ij} + 1}{\sum_{k} \left(N_{ik} + 1\right)}$$

**Aufgabe 3 (Code):** Fülle die Zählmatrix `N` mit den Trainingspaaren.

<details><summary>Tipp</summary>

`for x, y in zip(xs_train, ys_train):` liefert alle Paare. Ein Eintrag einer numpy-Matrix wird mit `N[x, y] += 1` erhöht.

*Für Profis:* `np.add.at(N, (xs_train, ys_train), 1)` erledigt dasselbe in einer Zeile. Die naheliegende Kurzform `N[xs_train, ys_train] += 1` ist dagegen **falsch**: Kommt ein Paar mehrfach vor, wird es trotzdem nur einmal gezählt. Diesen Unterschied brauchst du in Abschnitt 3 wieder.

</details>

In [ ]:
N = np.zeros((V, V), dtype=int)
# TODO (Aufgabe 3): für jedes Trainingspaar (x, y) den Eintrag N[x, y] um 1 erhöhen
...

print("Anzahl gezählter Paare:", N.sum())

In [ ]:
# TEST zu Aufgabe 3 (nur ausführen)
assert N.sum() == len(xs_train), "jedes Trainingspaar muss genau einmal gezählt werden"
assert N[0].sum() == len(train_namen), "Zeile '.': jeder Name beginnt genau einmal"
assert N[:, 0].sum() == len(train_namen), "Spalte '.': jeder Name endet genau einmal"
assert N[0, 0] == 0, "'.' direkt nach '.' gibt es nicht (keine leeren Namen)"
print("Alle Tests zu Aufgabe 3 bestanden.")

Die folgende (vorgegebene) Funktion zeigt die häufigsten Nachfolger eines Zeichens. `np.argsort` liefert die Indizes in aufsteigender Reihenfolge der Werte, `[::-1]` dreht sie um.

**Aufgabe 4 (Denkaufgabe):** Mit welchen Buchstaben beginnen sächsische Ortsnamen am häufigsten? Was folgt typischerweise auf `z`, `w`, `i` und `f`? Erkläre die Beobachtungen mit typischen Namensendungen. Probiere zwei weitere Zeichen aus.

In [ ]:
def zeige_nachfolger(zeichen, anzahl=6):
    """Zeigt die häufigsten Nachfolger eines Zeichens laut Zählmatrix N."""
    zeile = N[zeichen_zu_id[zeichen]]
    reihenfolge = np.argsort(zeile)[::-1]     # Indizes, absteigend nach Anzahl
    teile = []
    for j in reihenfolge[:anzahl]:
        teile.append(f"{id_zu_zeichen[int(j)]!r}: {zeile[j]}")
    print(f"nach {zeichen!r} ({zeile.sum()}-mal):", ", ".join(teile))

for z in [".", "z", "w", "i", "f"]:
    zeige_nachfolger(z)

**Aufgabe 5 (Code):** Berechne die Wahrscheinlichkeitsmatrix `P` mit +1-Glättung (Formel oben).

<details><summary>Tipp</summary>

`N + 1` addiert 1 zu jedem Eintrag. `M.sum(axis=1, keepdims=True)` liefert die Zeilensummen einer Matrix `M` als Spalte der Form (V, 1). Teilt man eine (V, V)-Matrix durch eine (V, 1)-Spalte, teilt numpy jede Zeile durch „ihre“ Summe (**Broadcasting**). Ohne `keepdims=True` würde stattdessen jede Spalte geteilt, und das wäre falsch.

</details>

In [ ]:
# TODO (Aufgabe 5): Wahrscheinlichkeitsmatrix P mit +1-Glättung berechnen
P = ...

In [ ]:
# TEST zu Aufgabe 5 (nur ausführen)
assert isinstance(P, np.ndarray) and P.shape == (V, V), "P muss ein V x V-Array sein"
assert np.allclose(P.sum(axis=1), 1.0), "jede Zeile von P muss die Summe 1 haben"
assert (P > 0).all(), "durch die Glättung darf kein Eintrag 0 sein"
i_s, i_c = zeichen_zu_id["s"], zeichen_zu_id["c"]
assert math.isclose(P[i_s, i_c], (N[i_s, i_c] + 1) / (N[i_s].sum() + V)), "Formel mit +1-Glättung prüfen"
print("P('c' | 's') =", round(P[i_s, i_c], 4))
print("Alle Tests zu Aufgabe 5 bestanden.")

### 2.2 Wie gut ist das Modell? Der Verlust

Aus 11.0 kennst du den Verlust $L = -\ln p_{\text{richtig}}$ für **eine** Vorhersage. Für einen ganzen Datensatz bilden wir den **Mittelwert** über alle Zeichenpaare (x, y):

$$L = \frac{1}{n} \sum_{i=1}^{n} -\ln P_{x_i\, y_i}$$

Je kleiner, desto besser. Zwei Vergleichswerte helfen beim Einordnen:

- Ein Modell, das **nichts** weiß, gibt jedem Zeichen die Wahrscheinlichkeit $\frac{1}{V}$ (Gleichverteilung). Sein Verlust ist $-\ln\frac{1}{V} = \ln V$, bei V = 30 etwa 3,4.
- $e^{L}$ heißt **Perplexität**: Das Modell ist im Mittel so unsicher, als müsste es zwischen $e^{L}$ gleich wahrscheinlichen Zeichen raten. Beim Gleichverteilungs-Modell sind das genau V Zeichen.

**Aufgabe 6 (Code):** Ergänze `verlust_zaehlmodell(P, namen)`.

<details><summary>Tipp</summary>

numpy kann mit zwei Index-Arrays viele Einträge auf einmal holen: `P[xs, ys]` ist das Array `[P[xs[0], ys[0]], P[xs[1], ys[1]], ...]`, also genau die Wahrscheinlichkeiten der richtigen Nachfolger. Danach brauchst du `np.log` und `np.mean`. Gib das Ergebnis mit `float(...)` als normale Zahl zurück.

</details>

In [ ]:
def verlust_zaehlmodell(P, namen):
    """Mittlerer Verlust -ln p über alle Zeichenpaare der Namen."""
    xs, ys = bigramme(namen)
    # TODO (Aufgabe 6): Wahrscheinlichkeiten der richtigen Nachfolger holen
    #                   und den Mittelwert von -ln p zurückgeben
    raise NotImplementedError("Aufgabe 6: verlust_zaehlmodell fehlt noch")

In [ ]:
# TEST zu Aufgabe 6 (nur ausführen)
gleichverteilung = np.full((V, V), 1 / V)
L_gleich = verlust_zaehlmodell(gleichverteilung, train_namen)
assert math.isclose(L_gleich, math.log(V), rel_tol=1e-9), "die Gleichverteilung muss den Verlust ln(V) haben"
L_zaehl_train = verlust_zaehlmodell(P, train_namen)
L_zaehl_test = verlust_zaehlmodell(P, test_namen)
assert L_zaehl_train < math.log(V), "das Zählmodell muss besser sein als Raten"
assert L_zaehl_test < math.log(V), "auch auf unbekannten Namen sollte es besser sein als Raten"
print(f"Gleichverteilung (Raten): {L_gleich:.3f}   (= ln {V})")
print(f"Zählmodell Training:      {L_zaehl_train:.3f}   Perplexität {math.exp(L_zaehl_train):.1f}")
print(f"Zählmodell Test:          {L_zaehl_test:.3f}   Perplexität {math.exp(L_zaehl_test):.1f}")
print("Alle Tests zu Aufgabe 6 bestanden.")

### 2.3 Namen erfinden

Jetzt lassen wir das Modell **autoregressiv** Namen erzeugen, genau wie in 11.0, Abschnitt 1: Wir beginnen mit `.`, ziehen das nächste Zeichen gewichtet zufällig (**Sampling**) nach den Wahrscheinlichkeiten in der passenden Zeile von `P`, hängen es an und machen mit dem neuen Zeichen weiter, bis `.` gezogen wird.

`rng.choice(V, p=wahrscheinlichkeiten)` zieht eine Zahl von 0 bis V−1, jede mit der angegebenen Wahrscheinlichkeit. Das ist das numpy-Gegenstück zu `random.choices(..., weights=...)` aus 11.0. Diese Funktion ist vorgegeben.

**Aufgabe 7 (Denkaufgabe):** Führe die Zelle mehrmals aus. Welche Namen klingen sächsisch, welche nicht? Woran liegt es, dass manche Namen viel zu lang, sehr kurz oder seltsam sind, obwohl jedes einzelne Buchstabenpaar „passt“?

In [ ]:
def generiere_zaehlmodell(P, max_laenge=20):
    """Erfindet einen Namen: Zeichen für Zeichen ziehen, bis '.' kommt."""
    name = ""
    aktuell = 0                                         # Start-Token "."
    while len(name) < max_laenge:
        naechstes = int(rng.choice(V, p=P[aktuell]))    # gewichtet zufällig ziehen
        if naechstes == 0:                              # Ende-Token gezogen
            break
        name = name + id_zu_zeichen[naechstes]
        aktuell = naechstes
    return name

for _ in range(15):
    print(generiere_zaehlmodell(P).capitalize())

## 3. Neuronales Bigramm-Modell mit Gradientenabstieg

Das Zählmodell funktioniert, weil es nur V = 30 mögliche Kontexte gibt. Ein echtes Sprachmodell betrachtet aber Tausende Tokens Kontext; so viele verschiedene Kontexte kann man nicht zählen. Deshalb benutzt man ein **neuronales Netz** mit Parametern, die durch **Gradientenabstieg** angepasst werden (11.8). Wir bauen das einfachste mögliche Netz, das dieselbe Aufgabe löst:

1. **Eingabe:** das aktuelle Zeichen x als **One-Hot-Vektor**: ein Vektor der Länge V mit einer 1 an der Stelle x und sonst 0.
2. **Eine Schicht ohne Aktivierungsfunktion:** Gewichtsmatrix `W` der Größe V×V. Die Logits sind $z = \text{onehot}(x) \cdot W$.
3. **Softmax** macht aus den Logits Wahrscheinlichkeiten $p$ für das nächste Zeichen.
4. **Verlust** $-\ln p_y$ für das richtige Zeichen y, gemittelt über alle Paare.

Das Netz hat V·V ≈ 900 Parameter. Am Anfang sind alle Gewichte 0.

### 3.1 One-Hot mal W ist eine Zeile von W

Multipliziert man einen One-Hot-Vektor mit W, bleibt genau **Zeile x** von W übrig, denn alle anderen Zeilen werden mit 0 multipliziert. Statt zu multiplizieren, können wir die Zeile also direkt nachschlagen: `W[x]`. Für viele Eingaben auf einmal liefert `W[xs]` eine Matrix mit einer Zeile pro Paar (Form `(n, V)`). Die Zelle zeigt das an zufälligen Gewichten (`@` ist in numpy das Matrixprodukt).

In [ ]:
W_demo = np.random.default_rng(0).normal(size=(V, V))   # zufällige Gewichte, nur zur Demonstration
x = zeichen_zu_id["a"]
onehot = np.zeros(V)
onehot[x] = 1.0
print(np.allclose(onehot @ W_demo, W_demo[x]))   # erwartet: True
print(W_demo[xs_train].shape)                    # erwartet: (Anzahl Trainingspaare, V)

### 3.2 Softmax und Verlust mit numpy

In 11.0 hast du Softmax mit Listen und `math.exp` programmiert. Mit numpy geht das für ganze Arrays auf einmal. Dabei gibt es eine Falle: `math.exp(1000)` ist zu groß für eine Kommazahl (Überlauf). Der Trick: Man zieht vor dem Exponenzieren das **Maximum** m der Logits ab. Am Ergebnis ändert das nichts, denn $e^{-m}$ kürzt sich heraus:

$$\frac{e^{z_i - m}}{\sum_k e^{z_k - m}} = \frac{e^{z_i}\cdot e^{-m}}{\left(\sum_k e^{z_k}\right)\cdot e^{-m}} = \frac{e^{z_i}}{\sum_k e^{z_k}}$$

Die Temperatur T wirkt wie in 11.0: Die Logits werden vor allem anderen durch T geteilt.

**Aufgabe 8 (Code):** Ergänze `softmax(logits, temperatur)`. Die Funktion soll für einen Vektor (Form `(V,)`) **und** für eine Matrix (Form `(n, V)`, jede Zeile einzeln) funktionieren.

<details><summary>Tipp 1</summary>

Drei Schritte: Maximum abziehen, `np.exp`, durch die Summe teilen. `np.max(z, axis=-1, keepdims=True)` und `np.sum(e, axis=-1, keepdims=True)` rechnen entlang der **letzten** Achse: bei einem Vektor über alle Einträge, bei einer Matrix für jede Zeile einzeln.

</details>

<details><summary>Tipp 2</summary>

`keepdims=True` sorgt dafür, dass das Ergebnis bei einer Matrix die Form `(n, 1)` hat. Dann zieht numpy von jeder Zeile ihr eigenes Maximum ab und teilt jede Zeile durch ihre eigene Summe (Broadcasting, wie in Aufgabe 5).

</details>

In [ ]:
def softmax(logits, temperatur=1.0):
    """Softmax mit Temperatur. logits: Vektor (V,) oder Matrix (n, V);
    bei einer Matrix wird jede Zeile einzeln in Wahrscheinlichkeiten umgerechnet."""
    z = logits / temperatur
    # TODO (Aufgabe 8): 1. Maximum abziehen  2. np.exp  3. durch die Summe teilen
    raise NotImplementedError("Aufgabe 8: softmax fehlt noch")

In [ ]:
# TEST zu Aufgabe 8 (nur ausführen)
p = softmax(np.array([4.0, 2.5, 1.5, 1.0]))
assert np.allclose(np.round(p, 3), [0.738, 0.165, 0.061, 0.037]), "Werte aus 11.0, Beispiel 3 (T = 1)"
p = softmax(np.array([4.0, 2.5, 1.5, 1.0]), 0.5)
assert np.allclose(np.round(p, 3), [0.944, 0.047, 0.006, 0.002]), "Werte aus 11.0, Beispiel 3 (T = 0,5)"
p = softmax(np.array([1000.0, 1000.0]))
assert np.allclose(p, [0.5, 0.5]), "nicht numerisch stabil: Maximum abziehen!"
m = softmax(np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]]))
assert m.shape == (2, 3) and np.allclose(m.sum(axis=1), 1.0), "bei einer Matrix muss jede Zeile die Summe 1 haben"
assert np.allclose(m[1], 1 / 3), "gleiche Logits ergeben gleiche Wahrscheinlichkeiten"
print("Alle Tests zu Aufgabe 8 bestanden.")

**Aufgabe 9 (Code):** Ergänze `verlust_netz(W, xs, ys)`: der mittlere Verlust des neuronalen Modells auf den Paaren (xs, ys).

<details><summary>Tipp</summary>

1. Logits aller Paare: `W[xs]` (Form `(n, V)`).
2. Wahrscheinlichkeiten: `softmax(...)` (Form `(n, V)`).
3. In Zeile i brauchst du die Spalte `ys[i]`: `p[np.arange(n), ys]` liefert genau diese n Werte (`np.arange(n)` ist das Array `[0, 1, …, n−1]`, n = `len(ys)`).
4. Mittelwert von `-np.log(...)`, als `float` zurückgeben.

</details>

In [ ]:
def verlust_netz(W, xs, ys):
    """Mittlerer Verlust -ln p(richtiges Zeichen) des neuronalen Modells mit Gewichten W."""
    # TODO (Aufgabe 9): 1. Logits W[xs]  2. softmax  3. p der richtigen Zeichen  4. Mittelwert von -ln p
    raise NotImplementedError("Aufgabe 9: verlust_netz fehlt noch")

In [ ]:
# TEST zu Aufgabe 9 (nur ausführen)
W0 = np.zeros((V, V))
assert math.isclose(verlust_netz(W0, xs_train, ys_train), math.log(V), rel_tol=1e-9), \
    "bei W = 0 muss der Verlust ln(V) sein"
# Mit W = ln(P) rechnet das Netz exakt wie das Zählmodell:
assert math.isclose(verlust_netz(np.log(P), xs_train, ys_train), L_zaehl_train, rel_tol=1e-9), \
    "mit W = ln(P) muss derselbe Verlust wie beim Zählmodell herauskommen"
print("Alle Tests zu Aufgabe 9 bestanden.")

**Aufgabe 10 (Denkaufgabe):** Der Test oben prüft zwei Aussagen. Begründe beide mit der Softmax-Formel.

1. Bei W = 0 ist der Verlust genau $\ln V$.
2. Mit $W = \ln P$ (Logarithmus jedes Eintrags) sagt das Netz exakt dieselben Wahrscheinlichkeiten vorher wie das Zählmodell. Was folgt daraus für die Frage, ob das neuronale Modell **besser** werden kann als das Zählmodell?

### 3.3 Der Gradient (Herleitung zum Nachvollziehen, nicht verlangt)

Beim Gradientenabstieg brauchen wir für jedes Gewicht die Ableitung des Verlusts. Für **ein** Paar mit den Logits $z$ und dem richtigen Zeichen y gilt, wie in der Kontrolle von 11.0, Beispiel 3:

$$L = -\ln p_y = -z_y + \ln\sum_k e^{z_k}$$

Ableiten nach einem Logit $z_j$ (Kettenregel beim zweiten Summanden):

$$\frac{\partial L}{\partial z_j} = -[j = y] + \frac{e^{z_j}}{\sum_k e^{z_k}} = p_j - [j = y]$$

Dabei ist $[j = y]$ gleich 1, wenn j das richtige Zeichen ist, sonst 0. In Worten: **Wahrscheinlichkeit minus One-Hot des richtigen Zeichens.** Das richtige Zeichen bekommt den Gradienten $p_y - 1 < 0$, sein Logit wird also beim Abstieg größer; alle anderen bekommen $p_j > 0$, ihre Logits werden kleiner. Je sicherer das Modell schon richtig liegt, desto kleiner die Korrektur.

Für den **Mittelwert** über n Paare wird jeder Beitrag durch n geteilt. Da die Logits von Paar i die Zeile `W[xs[i]]` sind, landet sein Gradient in genau dieser Zeile. Kommt ein Zeichen in vielen Paaren vor, müssen alle Beiträge **aufaddiert** werden. Das erledigt `np.add.at(dW, xs, d_logits)` (wie beim Zählen in Aufgabe 3: `dW[xs] += d_logits` würde mehrfach vorkommende Zeilen nur einmal erhöhen).

Die Funktion `gradient` ist vorgegeben. Die Testzelle danach prüft sie mit dem **Differenzenquotienten** aus der Mathematik: $\frac{L(w + h) - L(w - h)}{2h}$ für ein kleines h muss ungefähr die Ableitung ergeben. Dieser Test prüft damit auch deine Funktion `verlust_netz`.

In [ ]:
def gradient(W, xs, ys):
    """Gradient des mittleren Verlusts nach W (vorgegeben)."""
    n = len(xs)
    p = softmax(W[xs])                   # (n, V) Wahrscheinlichkeiten
    d_logits = p.copy()
    d_logits[np.arange(n), ys] -= 1.0    # p - onehot(y)
    d_logits = d_logits / n              # wegen des Mittelwerts
    dW = np.zeros_like(W)
    np.add.at(dW, xs, d_logits)          # Zeile xs[i] von dW bekommt d_logits[i] dazu
    return dW

In [ ]:
# TEST: Gradient mit dem Differenzenquotienten prüfen (nur ausführen)
W_test = np.random.default_rng(1).normal(0.0, 0.5, size=(V, V))
dW = gradient(W_test, xs_train, ys_train)
h = 1e-5
stellen = [(0, zeichen_zu_id["s"]), (zeichen_zu_id["e"], 0), (zeichen_zu_id["a"], zeichen_zu_id["u"])]
for (i, j) in stellen:
    W_plus = W_test.copy()
    W_plus[i, j] += h
    W_minus = W_test.copy()
    W_minus[i, j] -= h
    numerisch = (verlust_netz(W_plus, xs_train, ys_train) - verlust_netz(W_minus, xs_train, ys_train)) / (2 * h)
    print(f"W[{i}, {j}]: Formel {dW[i, j]:+.7f}   Differenzenquotient {numerisch:+.7f}")
    assert abs(numerisch - dW[i, j]) < 1e-6, "Gradient und Differenzenquotient passen nicht zusammen"
print("Gradient stimmt.")

### 3.4 Training

Gradientenabstieg heißt: Gewichte einen kleinen Schritt **gegen** den Gradienten verschieben, denn der Gradient zeigt in die Richtung, in der der Verlust am stärksten **steigt**:

$$W \leftarrow W - \text{Lernrate} \cdot \frac{\partial L}{\partial W}$$

Wir verwenden in jedem Schritt alle Trainingspaare auf einmal.

**Zur Lernrate:** Sie ist hier ungewöhnlich groß (10 bis 50). Der Grund: Der Verlust ist ein Mittelwert über rund 3000 Paare, und jede Zeile von W bekommt nur den Anteil der Paare ab, in denen ihr Zeichen vorkommt. Die Gradienten sind deshalb sehr klein.

**Zur Laufzeit:** In JupyterLite läuft Python im Browser (Pyodide) und ist einige Male langsamer als ein installiertes Python. 500 Schritte dauern dort einige Sekunden. Erhöhe die Schrittzahl nur schrittweise.

**Aufgabe 11 (Code):** Ergänze in `trainiere` den Aktualisierungsschritt.

<details><summary>Tipp</summary>

Eine Zeile genügt: `W = W - ...`. Der Gradient steht schon in `dW`, die Lernrate in `lernrate`. Die Zeile `raise NotImplementedError(...)` muss danach gelöscht werden.

</details>

In [ ]:
def trainiere(lernrate=30.0, schritte=500, ausgabe_alle=100):
    """Gradientenabstieg für das neuronale Bigramm-Modell. Gibt die trainierten Gewichte zurück."""
    W = np.zeros((V, V))
    for schritt in range(schritte):
        if schritt % ausgabe_alle == 0:
            print(f"Schritt {schritt:4d}: Verlust {verlust_netz(W, xs_train, ys_train):.4f}")
        dW = gradient(W, xs_train, ys_train)
        # TODO (Aufgabe 11): W einen Schritt der Größe lernrate gegen den Gradienten verschieben
        raise NotImplementedError("Aufgabe 11: Aktualisierung von W fehlt noch")
    print(f"Schritt {schritte:4d}: Verlust {verlust_netz(W, xs_train, ys_train):.4f}  (Ende)")
    return W

W = trainiere(lernrate=30.0, schritte=500)

In [ ]:
# TEST zu Aufgabe 11 (nur ausführen)
L_netz_train = verlust_netz(W, xs_train, ys_train)
L_netz_test = verlust_netz(W, xs_test, ys_test)
assert L_netz_train < math.log(V) - 0.5, "der Verlust ist kaum gesunken: Lernrate oder Schrittzahl erhöhen"
assert L_netz_train < L_zaehl_train + 0.15, "noch nicht nah genug am Zählmodell: länger trainieren"
print("              Training   Test")
print(f"Zählmodell     {L_zaehl_train:.3f}     {L_zaehl_test:.3f}")
print(f"neuronal       {L_netz_train:.3f}     {L_netz_test:.3f}")
print("Alle Tests zu Aufgabe 11 bestanden.")

**Aufgabe 12 (Experiment und Denkaufgabe):**

1. Führe die Zelle unten aus und notiere in einer Tabelle für jede Lernrate den Verlust nach 200 Schritten. Probiere danach selbst eine sehr kleine (z. B. 0,1) und eine sehr große Lernrate (z. B. 300) aus. Was beobachtest du? Vergleiche mit 11.8 (Hyperparameter Lernrate).
2. Erkläre, warum der Verlust des neuronalen Modells sich dem des Zählmodells nähert, ihn aber nicht deutlich unterbietet (Aufgabe 10).
3. Das Netz kann auf den Trainingsdaten sogar etwas **unter** dem Zählmodell landen, auf den Testdaten aber etwas darüber. Erkläre das mit der +1-Glättung.

In [ ]:
# Experiment: verschiedene Lernraten mit je 200 Schritten (dauert in JupyterLite etwas)
for lr in [1.0, 10.0, 30.0, 60.0]:
    print("Lernrate", lr)
    trainiere(lernrate=lr, schritte=200, ausgabe_alle=100)
    print()

## 4. Generieren mit Temperatur

### 4.1 Die Funktion `generiere`

**Aufgabe 13 (Code):** Schreibe `generiere(W, temperatur, max_laenge)`: Sie funktioniert wie `generiere_zaehlmodell` aus Abschnitt 2.3, aber die Wahrscheinlichkeiten kommen aus dem neuronalen Modell, und zwar mit Temperatur (11.0, Abschnitt 5): $p = \text{softmax}(W[\text{aktuell}],\ T)$.

<details><summary>Tipp</summary>

Kopiere den Rumpf von `generiere_zaehlmodell` und ersetze `P[aktuell]` durch `softmax(W[aktuell], temperatur)`. Denke an `int(...)` um das Ergebnis von `rng.choice`.

</details>

In [ ]:
def generiere(W, temperatur=1.0, max_laenge=20):
    """Erfindet einen Namen mit dem neuronalen Modell W."""
    # TODO (Aufgabe 13): wie generiere_zaehlmodell, aber mit den Wahrscheinlichkeiten
    #                    softmax(Logits des aktuellen Zeichens, temperatur)
    raise NotImplementedError("Aufgabe 13: generiere fehlt noch")

In [ ]:
# TEST zu Aufgabe 13 (nur ausführen)
i_a = zeichen_zu_id["a"]
W_a = np.full((V, V), -50.0)
W_a[0, i_a] = 0.0       # nach "." praktisch sicher "a"
W_a[i_a, 0] = 0.0       # nach "a" praktisch sicher "."
assert generiere(W_a) == "a", "mit W_a muss immer 'a' herauskommen"
W_aa = np.full((V, V), -50.0)
W_aa[0, i_a] = 0.0
W_aa[i_a, i_a] = 0.0    # nach "a" praktisch sicher wieder "a"
assert generiere(W_aa, max_laenge=7) == "aaaaaaa", "max_laenge wird nicht beachtet"
for _ in range(20):
    name = generiere(W, 1.0)
    assert isinstance(name, str) and "." not in name and len(name) <= 20
print("Alle Tests zu Aufgabe 13 bestanden.")

### 4.2 Experiment mit der Temperatur

**Aufgabe 14 (Experiment und Denkaufgabe):** Führe die Zelle mehrmals aus. Beschreibe, wie sich die Namen bei T = 0,5, T = 1 und T = 2 unterscheiden (Länge, typische Bausteine, Wiederholungen, „Buchstabensalat“). Erkläre die Unterschiede mit der Wirkung der Temperatur auf die Softmax (11.0, Abschnitt 5). Welche Temperatur würdest du für einen „Ortsnamen-Generator“ für ein Fantasy-Spiel in Sachsen wählen?

In [ ]:
for T in [0.5, 1.0, 2.0]:
    namen = [generiere(W, T).capitalize() for _ in range(12)]
    print(f"T = {T}:", ", ".join(namen))
    print()

### 4.3 Abschreiben oder erfinden?

Erzeugt das Modell eigentlich echte Ortsnamen? Und wenn ja: Hat es sie abgeschrieben?

**Aufgabe 15 (Code):** Schreibe `zaehle_echte(erfundene, liste)`: Sie gibt zurück, wie viele Einträge der Liste `erfundene` in `liste` vorkommen (doppelte Einträge in `erfundene` zählen mehrfach).

<details><summary>Tipp</summary>

Wandle `liste` zuerst mit `set(liste)` in eine Menge um. Die Abfrage `name in menge` ist bei einer Menge viel schneller als bei einer Liste (vgl. Laufzeit von Suchverfahren). Danach genügt eine Schleife mit Zähler.

</details>

In [ ]:
def zaehle_echte(erfundene, liste):
    """Wie viele Einträge von erfundene kommen in liste vor? (doppelte zählen mehrfach)"""
    # TODO (Aufgabe 15)
    raise NotImplementedError("Aufgabe 15: zaehle_echte fehlt noch")

In [ ]:
# TEST zu Aufgabe 15 (nur ausführen)
assert zaehle_echte(["dresden", "xyz", "dresden"], ["dresden", "leipzig"]) == 2
assert zaehle_echte([], ["pirna"]) == 0
assert zaehle_echte(["pirna"], []) == 0
print("Alle Tests zu Aufgabe 15 bestanden.")

In [ ]:
anzahl = 500
orte_menge = set(orte)
for T in [0.5, 1.0, 2.0]:
    erfunden = [generiere(W, T) for _ in range(anzahl)]
    im_training = zaehle_echte(erfunden, train_namen)
    im_test = zaehle_echte(erfunden, test_namen)
    verschieden = len(set(erfunden))
    beispiele = sorted(set(n for n in erfunden if n in orte_menge))[:8]
    print(f"T = {T}: {im_training} im Trainingsteil, {im_test} im Testteil, "
          f"{verschieden} verschiedene unter {anzahl}; echte z. B.: {beispiele}")

**Aufgabe 16 (Denkaufgabe):**

1. Welche echten Namen erzeugt das Modell? Was haben sie gemeinsam?
2. Manche erzeugten Namen stehen im **Testteil**, also in Daten, die das Modell nie gesehen hat. Hat es sie „abgeschrieben“? Erkläre, wie das möglich ist.
3. Kann ein Bigramm-Modell einen langen Namen wie „johanngeorgenstadt“ auswendig lernen? Begründe.
4. Ein erfundener Name wie „Großwitzschen“ klingt echt. Vergleiche mit einer **Halluzination** eines großen Sprachmodells (11.0, Abschnitt 7). Achtung: Manche „erfundenen“ Namen gibt es in Sachsen wirklich, nur nicht in unserer Liste. Was bedeutet das für die Frage, ob ein erzeugter Name „falsch“ ist?

## 5. Erweiterungen (Auswahl)

Wähle mindestens eine Erweiterung. Sie sind nach Schwierigkeit geordnet. Erweiterung d ist ein **Zusatz** für sehr schnelle Gruppen.

### 5a. Trigramm-Zählmodell: mehr Kontext und Overfitting

Das Trigramm-Modell sieht die **letzten zwei** Zeichen. Die Zählmatrix wird dreidimensional: `N3[a, b, c]` zählt, wie oft Zeichen c auf das Paar (a, b) folgt. Am Namensanfang füllen wir mit zwei Punkten auf: aus „aue“ wird `..aue.` mit den Dreiergruppen `..a`, `.au`, `aue`, `ue.`.

`N3` hat V·V·V ≈ 27 000 Einträge, es gibt aber nur rund 3000 Trainingsdreiergruppen. Die meisten Kontexte (a, b) kommen also nie vor. Deshalb wird die Glättung wichtig. Statt +1 verwenden wir einen Glättungsparameter α (alpha):

$$P_{abc} = \frac{N_{abc} + \alpha}{\sum_k \left(N_{abk} + \alpha\right)}$$

**Erweiterung a (Code):** Ergänze die drei Funktionen. Führe dann das Experiment aus und beantworte:

1. Wie verändern sich Trainings- und Testverlust, wenn α kleiner wird? Erkläre mit dem Begriff **Overfitting** (11.3).
2. Ist das Trigramm-Modell auf den Testdaten besser als das Bigramm-Modell? Für welches α?
3. Vergleiche die erzeugten Namen mit denen des Bigramm-Modells.

<details><summary>Tipp zu trigramme</summary>

Wie `bigramme`, nur mit drei Listen und `range(len(folge) - 2)`. Die Dreiergruppe an Position i ist `folge[i]`, `folge[i + 1]`, `folge[i + 2]`.

</details>

<details><summary>Tipp zu trigramm_modell und verlust_trigramm</summary>

Zählen: `np.add.at(N3, (a, b, c), 1)` (oder eine Schleife mit `N3[x, y, z] += 1`). Normieren wie in Aufgabe 5, aber die Summe über die **letzte** Achse: `.sum(axis=2, keepdims=True)`. Im Verlust holt `P3[a, b, c]` alle benötigten Wahrscheinlichkeiten auf einmal.

</details>

In [ ]:
def trigramme(namen):
    """Liefert drei Arrays: Zeichen c (drittes Array) folgt auf das Paar (a, b) (erstes, zweites Array)."""
    erste, zweite, dritte = [], [], []
    for name in namen:
        folge = ".." + name + "."
        # TODO (Erweiterung a): alle Dreiergruppen von folge durchgehen und die IDs anhängen
        ...
    return np.array(erste, dtype=int), np.array(zweite, dtype=int), np.array(dritte, dtype=int)

def trigramm_modell(namen, alpha=1.0):
    """Wahrscheinlichkeiten P3[a, b, c] mit alpha-Glättung."""
    a, b, c = trigramme(namen)
    N3 = np.zeros((V, V, V))
    # TODO (Erweiterung a): zählen, glätten, normieren
    raise NotImplementedError("Erweiterung a: trigramm_modell fehlt noch")

def verlust_trigramm(P3, namen):
    """Mittlerer Verlust -ln p des Trigramm-Modells."""
    # TODO (Erweiterung a)
    raise NotImplementedError("Erweiterung a: verlust_trigramm fehlt noch")

In [ ]:
# TEST zu Erweiterung a (nur ausführen)
a, b, c = trigramme(["aue"])
assert [id_zu_zeichen[int(i)] for i in a] == list("..au")
assert [id_zu_zeichen[int(i)] for i in b] == list(".aue")
assert [id_zu_zeichen[int(i)] for i in c] == list("aue.")
P3 = trigramm_modell(train_namen, alpha=1.0)
assert P3.shape == (V, V, V) and np.allclose(P3.sum(axis=2), 1.0), "jede Verteilung P3[a, b] muss die Summe 1 haben"
assert math.isclose(verlust_trigramm(np.full((V, V, V), 1 / V), train_namen), math.log(V), rel_tol=1e-9)
print("Alle Tests zu Erweiterung a bestanden.")

In [ ]:
print("alpha      Training   Test")
for alpha in [1.0, 0.3, 0.1, 0.01, 0.001]:
    P3 = trigramm_modell(train_namen, alpha)
    print(f"{alpha:<10} {verlust_trigramm(P3, train_namen):.3f}      {verlust_trigramm(P3, test_namen):.3f}")
print(f"Bigramm    {L_zaehl_train:.3f}      {L_zaehl_test:.3f}")

def generiere_trigramm(P3, max_laenge=20):
    """Erfindet einen Namen mit dem Trigramm-Modell (vorgegeben)."""
    name = ""
    a, b = 0, 0                                   # Kontext ".."
    while len(name) < max_laenge:
        c = int(rng.choice(V, p=P3[a, b]))
        if c == 0:
            break
        name = name + id_zu_zeichen[c]
        a, b = b, c
    return name

P3 = trigramm_modell(train_namen, alpha=0.1)
print()
print(", ".join(generiere_trigramm(P3).capitalize() for _ in range(15)))

### 5b. Top-k-Sampling

Wie in Aufgabe 11.0.4: Statt aus allen V Zeichen wird nur unter den **k wahrscheinlichsten** gezogen; deren Wahrscheinlichkeiten werden neu normiert, damit sie wieder die Summe 1 haben.

**Erweiterung b (Code):** Ergänze `top_k_verteilung(p, k)` und `generiere_top_k(W, k, temperatur, max_laenge)`. Untersuche dann: Was passiert bei k = 1? Wie wirkt k im Vergleich zur Temperatur?

<details><summary>Tipp</summary>

`np.argsort(p)[::-1][:k]` liefert die Indizes der k größten Werte. Lege mit `np.zeros_like(p)` ein Array voller Nullen an, übertrage an diesen Indizes die Werte aus `p` und teile durch die Summe. In `generiere_top_k` wendest du `top_k_verteilung` auf `softmax(W[aktuell], temperatur)` an, bevor du ziehst.

</details>

In [ ]:
def top_k_verteilung(p, k):
    """Behält nur die k größten Wahrscheinlichkeiten, setzt den Rest auf 0 und normiert neu (k >= 1)."""
    # TODO (Erweiterung b)
    raise NotImplementedError("Erweiterung b: top_k_verteilung fehlt noch")

def generiere_top_k(W, k=5, temperatur=1.0, max_laenge=20):
    """Wie generiere, aber es wird nur unter den k wahrscheinlichsten Zeichen gezogen."""
    # TODO (Erweiterung b)
    raise NotImplementedError("Erweiterung b: generiere_top_k fehlt noch")

In [ ]:
# TEST zu Erweiterung b (nur ausführen)
q = top_k_verteilung(np.array([0.1, 0.5, 0.4]), 2)
assert np.allclose(q, [0.0, 5 / 9, 4 / 9]), "k = 2: nur 0.5 und 0.4 bleiben, neu normiert"
assert np.allclose(top_k_verteilung(np.array([0.1, 0.5, 0.4]), 1), [0.0, 1.0, 0.0])
assert np.allclose(top_k_verteilung(np.array([0.1, 0.5, 0.4]), 10), [0.1, 0.5, 0.4]), "k größer als die Länge: alles bleibt"
q = top_k_verteilung(softmax(np.array([4.0, 2.5, 1.5, 1.0])), 2)
assert np.allclose(np.round(q, 3), [0.818, 0.182, 0.0, 0.0]), "Werte aus Aufgabe 11.0.4"
assert generiere_top_k(W_a, k=1) == "a"
print("Alle Tests zu Erweiterung b bestanden.")

for k in [1, 3, 5, V]:
    print(f"k = {k}:", ", ".join(generiere_top_k(W, k).capitalize() for _ in range(10)))

### 5c. Eigener Datensatz

Das Modell ist nicht auf Ortsnamen festgelegt. Ersetze in Abschnitt 1 die Daten durch eine eigene Liste und führe alle Zellen erneut aus (*Run → Run All Cells*). Ideen:

- Vornamen aus einer **öffentlichen Statistik** (z. B. beliebteste Vornamen eines Jahrgangs),
- Pokémon-Namen, Namen von Sternen, Dinosauriern oder Pflanzen,
- erfundene Gewässernamen, Straßennamen deines Ortes, Namen von Bergen im Erzgebirge.

**Wichtig:** Verwende **keine personenbezogenen Daten**, also z. B. keine Namensliste deiner Klasse oder deines Vereins, auch nicht „nur zum Testen“ (vgl. 11.0, Abschnitt 8).

Hinweise:

- Enthalten deine Namen Leerzeichen oder Bindestriche, funktioniert `orte_roh.lower().split()` nicht mehr. Schreibe dann **einen Namen pro Zeile** und verwende `orte = sorted(set(z.strip().lower() for z in orte_roh.split("\n") if z.strip()))`. Leerzeichen und Bindestrich werden dann einfach weitere Zeichen im Vokabular. Der Punkt `.` darf in den Namen nicht vorkommen.
- Mit weniger als etwa 150 Namen lernt das Modell wenig. Vergleiche den Verlust mit $\ln V$.
- Die Tests in Aufgabe 2, 5 und 13 verwenden die Buchstaben a, c, e, i, n, p, r, s, u. Fehlt einer davon in deinen Daten, schlägt der Test fehl, obwohl dein Code stimmt.

### 5d. Zusatz für sehr schnelle Gruppen: neuronales Netz mit Kontext 3 und Embeddings

Dieser Teil folgt einer Idee von Yoshua Bengio (2003), aus der die heutigen Sprachmodelle hervorgegangen sind. Der **Code ist vollständig vorgegeben**, auch die Backpropagation. Deine Aufgabe ist es, ihn zu verstehen und mit den **Hyperparametern** zu experimentieren.

**Aufbau des Netzes** (k = 3 Zeichen Kontext, d = Länge eines Embeddings, h = Neuronen der versteckten Schicht):

| Schritt | Rechnung | Form bei n Beispielen |
|---|---|---|
| Eingabe | IDs der letzten 3 Zeichen, z. B. `..p` → `[0, 0, 17]` | (n, 3) |
| Embedding | jede ID wird durch ihre Zeile der Matrix **C** ersetzt (11.0, Abschnitt 3: Embeddings werden gelernt) | (n, 3, d) |
| Nebeneinanderlegen | die 3 Embeddings werden zu einem Vektor | (n, 3·d) |
| versteckte Schicht | $\tanh(e \cdot W_1 + b_1)$, Aktivierungsfunktion tanh (Werte zwischen −1 und 1) | (n, h) |
| Ausgabeschicht | Logits $= \text{hid} \cdot W_2 + b_2$ | (n, V) |
| Softmax, Verlust | wie in Abschnitt 3 | (n, V) |

**Backpropagation** wendet die Kettenregel von hinten nach vorn an: Der Gradient der Logits ist wie in Abschnitt 3.3 $p - \text{onehot}(y)$ (geteilt durch n). Von dort wird er durch $W_2$, die Ableitung von tanh ($1 - \tanh^2$) und $W_1$ zurück bis zu den Embeddings gereicht. Die Embedding-Gradienten werden wie in Abschnitt 3 mit `np.add.at` in die Zeilen von C einsortiert.

**Minibatch:** Statt in jedem Schritt alle Paare zu verwenden, nimmt das Training jeweils 64 zufällig gewählte Beispiele. Das ist ungenauer, aber viel schneller; so trainieren auch große Modelle. Gegen Ende wird die Lernrate verkleinert, damit sich die Gewichte „setzen“.

In [ ]:
K = 3   # Kontextlänge: so viele vorherige Zeichen sieht das Netz

def kontext_daten(namen, k=K):
    """X: (n, k) IDs der letzten k Zeichen, Y: (n,) ID des jeweils nächsten Zeichens."""
    X, Y = [], []
    for name in namen:
        kontext = [0] * k                  # am Anfang nur "."
        for z in name + ".":
            ziel = zeichen_zu_id[z]
            X.append(kontext)
            Y.append(ziel)
            kontext = kontext[1:] + [ziel]   # Fenster um ein Zeichen weiterschieben
    return np.array(X, dtype=int), np.array(Y, dtype=int)

X_train, Y_train = kontext_daten(train_namen)
X_test, Y_test = kontext_daten(test_namen)
print(X_train.shape, Y_train.shape)
for i in range(6):
    print("".join(id_zu_zeichen[int(j)] for j in X_train[i]), "->", id_zu_zeichen[int(Y_train[i])])

In [ ]:
def init_mlp(d=8, h=64, k=K, seed=1):
    """Zufällige Startwerte für alle Parameter."""
    g = np.random.default_rng(seed)
    return {
        "C":  g.normal(0.0, 1.0, size=(V, d)),                       # Embeddings
        "W1": g.normal(0.0, 1.0, size=(k * d, h)) / math.sqrt(k * d),
        "b1": np.zeros(h),
        "W2": g.normal(0.0, 1.0, size=(h, V)) * 0.01,                # klein: Startverlust etwa ln(V)
        "b2": np.zeros(V),
    }

def mlp_vorwaerts(params, X):
    """Vorwärtsrechnung; liefert alle Zwischenergebnisse für die Backpropagation."""
    emb = params["C"][X]                              # (n, k, d)
    e = emb.reshape(len(X), -1)                       # (n, k*d)
    hid = np.tanh(e @ params["W1"] + params["b1"])    # (n, h)
    logits = hid @ params["W2"] + params["b2"]        # (n, V)
    return emb, e, hid, logits

def mlp_verlust(params, X, Y):
    logits = mlp_vorwaerts(params, X)[3]
    p = softmax(logits)
    return float(np.mean(-np.log(p[np.arange(len(Y)), Y])))

def mlp_gradienten(params, X, Y):
    """Backpropagation (vorgegeben): Kettenregel rückwärts durch alle Schichten."""
    n = len(X)
    emb, e, hid, logits = mlp_vorwaerts(params, X)
    d_logits = softmax(logits)
    d_logits[np.arange(n), Y] -= 1.0
    d_logits = d_logits / n                          # wie in Abschnitt 3.3
    grads = {}
    grads["W2"] = hid.T @ d_logits                   # (h, V)
    grads["b2"] = d_logits.sum(axis=0)               # (V,)
    d_hid = d_logits @ params["W2"].T                # (n, h)
    d_vor = d_hid * (1.0 - hid ** 2)                 # Ableitung von tanh
    grads["W1"] = e.T @ d_vor                        # (k*d, h)
    grads["b1"] = d_vor.sum(axis=0)                  # (h,)
    d_e = d_vor @ params["W1"].T                     # (n, k*d)
    grads["C"] = np.zeros_like(params["C"])
    np.add.at(grads["C"], X, d_e.reshape(emb.shape))   # Embedding-Gradienten einsortieren
    return grads

params = init_mlp()
print("Parameter:", sum(wert.size for wert in params.values()))
print("Startverlust:", round(mlp_verlust(params, X_train, Y_train), 3), " ln(V) =", round(math.log(V), 3))

In [ ]:
# TEST: Backpropagation mit dem Differenzenquotienten prüfen (nur ausführen)
params_test = init_mlp(d=4, h=10, seed=3)
params_test["W2"] = np.random.default_rng(4).normal(0.0, 0.5, size=params_test["W2"].shape)
Xk, Yk = X_train[:50], Y_train[:50]
grads = mlp_gradienten(params_test, Xk, Yk)
eps = 1e-5
for name in ["C", "W1", "b1", "W2", "b2"]:
    stelle = (0,) * params_test[name].ndim
    alt = params_test[name][stelle]
    params_test[name][stelle] = alt + eps
    L_plus = mlp_verlust(params_test, Xk, Yk)
    params_test[name][stelle] = alt - eps
    L_minus = mlp_verlust(params_test, Xk, Yk)
    params_test[name][stelle] = alt
    numerisch = (L_plus - L_minus) / (2 * eps)
    print(f"{name:3s} {stelle}: Backpropagation {grads[name][stelle]:+.7f}   Differenzenquotient {numerisch:+.7f}")
    assert abs(numerisch - grads[name][stelle]) < 1e-6, "Backpropagation passt nicht zum Differenzenquotienten"
print("Backpropagation stimmt.")

In [ ]:
def trainiere_mlp(params, schritte=4000, lernrate=0.2, batch=64, ausgabe_alle=500, seed=5):
    """Minibatch-Gradientenabstieg; im letzten Viertel mit einem Zehntel der Lernrate."""
    g = np.random.default_rng(seed)
    for schritt in range(schritte):
        if schritt % ausgabe_alle == 0:
            print(f"Schritt {schritt:5d}: Training {mlp_verlust(params, X_train, Y_train):.3f}   "
                  f"Test {mlp_verlust(params, X_test, Y_test):.3f}")
        auswahl = g.integers(0, len(X_train), size=batch)    # zufälliger Minibatch
        grads = mlp_gradienten(params, X_train[auswahl], Y_train[auswahl])
        lr = lernrate if schritt < 0.75 * schritte else lernrate / 10
        for name in params:
            params[name] -= lr * grads[name]
    print(f"Ende ({schritte} Schritte): Training {mlp_verlust(params, X_train, Y_train):.3f}   "
          f"Test {mlp_verlust(params, X_test, Y_test):.3f}")
    return params

params = trainiere_mlp(init_mlp(d=8, h=64), schritte=4000, lernrate=0.2)
assert mlp_verlust(params, X_train, Y_train) < math.log(V) - 0.5, "das Training hat kaum etwas gebracht"
print(f"zum Vergleich Bigramm-Zählmodell: Training {L_zaehl_train:.3f}   Test {L_zaehl_test:.3f}")

In [ ]:
def generiere_mlp(params, temperatur=1.0, max_laenge=20, k=K):
    """Erfindet einen Namen mit dem Netz (vorgegeben)."""
    name = ""
    kontext = [0] * k
    while len(name) < max_laenge:
        logits = mlp_vorwaerts(params, np.array([kontext]))[3][0]
        naechstes = int(rng.choice(V, p=softmax(logits, temperatur)))
        if naechstes == 0:
            break
        name = name + id_zu_zeichen[naechstes]
        kontext = kontext[1:] + [naechstes]
    return name

erfunden = [generiere_mlp(params, 0.8) for _ in range(300)]
print(", ".join(n.capitalize() for n in erfunden[:20]))
print(zaehle_echte(erfunden, train_namen), "von 300 stehen im Trainingsteil,",
      zaehle_echte(erfunden, test_namen), "im Testteil")

**Experimente (Zusatz):** Lege eine Tabelle an mit den Spalten d, h, Lernrate, Schritte, Batch, Parameterzahl, Trainingsverlust, Testverlust und trage mindestens fünf Versuche ein. Untersuche zum Beispiel:

1. Was passiert bei sehr großem h (z. B. 300)? Achte auf den **Abstand** zwischen Trainings- und Testverlust (Overfitting bei nur rund 300 Trainingsnamen).
2. Wie wirkt sich eine größere Kontextlänge `K` aus (oben `K = 4` setzen und **alle** Zellen von 5d neu ausführen)?
3. Ab welcher Lernrate wird das Training instabil?
4. Vergleiche den besten Testverlust mit Bigramm- und Trigramm-Modell. Lohnt sich das neuronale Netz bei so wenigen Daten?

Beachte die Laufzeit in JupyterLite: Verdopple die Schrittzahl erst, wenn ein Versuch weniger als etwa 30 Sekunden gedauert hat.

## 6. Reflexion

Die Zelle zeigt, was das neuronale Bigramm-Modell gelernt hat: für jedes Zeichen die drei wahrscheinlichsten Nachfolger.

In [ ]:
for z in vokabular:
    p = softmax(W[zeichen_zu_id[z]])
    beste = np.argsort(p)[::-1][:3]
    teile = [f"{id_zu_zeichen[int(j)]!r} {p[j]:.2f}" for j in beste]
    print(f"{z!r} ->", ", ".join(teile))

**Vergleich mit einem echten Sprachmodell**

| | unser Modell | großes Sprachmodell (z. B. hinter schulKI) |
|---|---|---|
| Token | ein Zeichen, V ≈ 30 | Wortteile (Byte-Pair-Encoding), V ≈ 50 000 bis 200 000 |
| Kontext | 1 Zeichen (Zusatz 5d: 3) | Tausende bis Hunderttausende Tokens, verarbeitet mit **Attention** |
| Parameter | V·V ≈ 900 (Zusatz 5d: einige Tausend) | Milliarden |
| Trainingsdaten | rund 300 Namen, etwa 3000 Zeichenpaare | Billionen Tokens |
| Training | Gradientenabstieg, Sekunden im Browser | Gradientenabstieg, Wochen auf Tausenden Grafikprozessoren |
| nach dem Vortraining | nichts | Feinabstimmung mit Dialogen, RLHF, Systemprompt |
| **gleich** | nächstes Token vorhersagen, Softmax mit Temperatur, Verlust $-\ln p$, Gradientenabstieg, autoregressives Erzeugen | ebenso |

**Aufgabe R (Denkaufgabe):** Beantworte in einer Markdown-Zelle:

1. Was genau hat dein Modell gelernt, und was **nicht**? Nutze die Ausgabe oben und Begriffe aus diesem Notebook (Bigramm, Wahrscheinlichkeit, Kontext).
2. Nenne drei Gründe, warum ein großes Sprachmodell viel bessere Texte schreibt als unser Modell. Ordne jeden Grund einer Zeile der Tabelle zu.
3. Erkläre an deinem Modell, warum Sprachmodelle **halluzinieren**: Warum erzeugt es selbstbewusst Namen von Orten, die es nicht gibt, und warum kann es selbst nicht erkennen, welche seiner Namen echt sind?
4. Unsere Daten enthalten viele Namen auf *-dorf* und *-itz*. Was würde passieren, wenn fast nur Namen aus der Oberlausitz in den Daten stünden? Übertrage die Beobachtung auf **Bias** in großen Sprachmodellen.
5. Projektdokumentation: Halte fest, welche Teile du selbst geschrieben, welche du übernommen und wofür du gegebenenfalls KI genutzt hast (Kennzeichnung wie in 11.0, Abschnitt 9).